In [1]:
import pandas as pd

df = pd.read_csv('./data/raw_data/merged_data.csv')
df.head(5)

,date,occupation,channel,place,gender,ageRange,question_raw,answer_raw
0,20230116,ARD,MOCK,ONLINE,FEMALE,-34,면접자께서는 지금부터 삼 년 뒤까지 삼 년 내 주요 성장 계획이나 목표가 있으십니까...,네 저는 디자이너로서 삼 년 안에 저만의 브랜드를 만드는 게 목표입니다. 어 아직까...
1,20230116,ARD,MOCK,ONLINE,FEMALE,-34,업무를 하다가 만일 상사가 지원자님에게 어려운 지시를 준다면 어떻게 극복하시겠어요,네 디자인을 하다가 제가 이제 메인 디자이너 상사께 받은 여러 가지 지시 사항이 있...
2,20230116,ARD,MOCK,ONLINE,FEMALE,-34,그 이전 직장에서 그 퇴사를 하셨는데요 뭐 퇴사하신 것을 혹시라도 후회하신 적은 없...,네 저는 전에 직장에서 일할 수 있었다는 걸 굉장히 감사하게 생각을 하고요. 퇴사한...
3,20230116,ARD,MOCK,ONLINE,FEMALE,-34,주로 영감은 어디에서 찾으시나요,네 디자인 일을 할 때는 아무래도 영감 그리고 아이디어를 얻는 일이 참 중요합니다....
4,20230116,ARD,MOCK,ONLINE,FEMALE,-34,일을 하다 보면 상사와 트러블이 생길 수가 있는데요 어떻게 하면 좋을까요 또 상사가...,네 일단 업무를 하다보면 어 상사와 트러블이 발생할 수도 있을 것 같습니다. 그리고...


In [2]:
df_ict = df[df['occupation'] == 'ICT'] 
df_rnd = df[df['occupation'] == 'RND']

In [3]:
df = pd.concat([df_ict, df_rnd], ignore_index=True)
df = df[['question_raw','answer_raw']]
df.to_csv('./data/data_preprocessing/sorted_data.csv', index=False, encoding='utf-8-sig')

In [4]:
df = pd.read_csv('./data/data_preprocessing/sorted_data.csv')
df.head(5)

,question_raw,answer_raw
0,지원자님 일을 하다 보면 영어로 서류를 작성해야 할 일이 생길 수 있습니다 지원자님...,네 가능합니다. 영어로 서류를 작성해야 할 시에 할 시를 대비하기 위해서 어 많은 ...
1,커뮤니케이션을 잘 할 수 있는 본인만의 스킬이 있나요 있다면 저희에게 한 번 직접 ...,커뮤니케이션을 잘 할 수 있는 저만의 스킬은 소소하게 스물 토크를 통해서 하는 것인...
2,어 지원장님은 의견을 주장해야 될 때 강하게 주장을 하시나요 아니면 약하게 하시나요...,의견을 주장할 때 원래는 강하게 하는 편이었는데 요즘은 강하게 하기 보다는 적당하게...
3,지금까지 살아오시면서 가장 힘들었던 경험에는 어떤 것이 있는지요 그 경험도 말씀해 ...,지금까지 살아오면서 가장 힘들었던 경험은 대학 입시에서 과를 결정하는 데 부모님과 ...
4,면접자분이 역사 속에서 가장 존경하는 일 인물을 말씀해 주시고 그리고 왜 그 인물을...,역사 속에서 가장 존경하는 인물은 성경에 나오는 인물인 모세인데요. 모세는 이스라엘...


In [5]:
# 한글 아닌 데이터 제거
df['question_raw'] = df['question_raw'].replace(r'[^0-9가-힣ㄱ-ㅎㅏ-ㅣ\s]', '', regex=True)
df['answer_raw'] = df['answer_raw'].replace(r'[^0-9가-힣ㄱ-ㅎㅏ-ㅣ\s]', '', regex=True)

In [6]:
from tqdm import tqdm
from konlpy.tag import Okt
# def load_ko_stopwords(filepath):
#     with open(filepath, 'r', encoding='UTF-8') as f:
#         return [line.strip() for line in f]

okt = Okt()

def load_ko_stopwords(filepath):
    with open(filepath, 'r', encoding='UTF-8') as f:
        return [line.strip() for line in f]
    
def get_core_sentence(text):
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    
    if len(sentences) >= 2:
        return sentences[0] + '. ' + sentences[-1] + '.'
    elif len(sentences) == 1:
        return sentences[0] + '.'
    return text

def preprocess(sentences, ko_stopwords, answer=False):
    result = []
    for sentence in tqdm(sentences):
        if answer:
            sentence = get_core_sentence(sentence)
        tokens = okt.morphs(sentence)

        if answer:
            tokens = ['<sos>'] + tokens + ['<eos>']
        else:
            tokens = tokens + ['<eos>']

        tokens = [token for token in tokens if token not in ko_stopwords]
        result.append(tokens)
    return result

ko_stopwords = load_ko_stopwords('./ko_stopwords.txt')
question = preprocess(df['question_raw'], ko_stopwords)
answer = preprocess(df['answer_raw'], ko_stopwords)

100%|██████████| 11926/11926 [01:49<00:00, 109.32it/s]


In [7]:
import torch
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

BATCH_SIZE = 64                # 배치 크기
MAX_VOCAB_SIZE = 10000         # 단어사전 크기
EMBEDDING_DIM = 300            # 임베딩 차원
LATENT_DIM = 512               # 뉴런 수

In [8]:
# 토크나이저, 정수 시퀀스 변환
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<UNK>')
tokenizer.fit_on_texts(question + answer)

question_seq = tokenizer.texts_to_sequences(question)
answer_seq = tokenizer.texts_to_sequences(answer)

In [9]:
# 최대 문장 길이 확인
num_words = len(tokenizer.word_index) + 1
q_max_len = max(len(s) for s in question_seq)
a_max_len = max(len(s) for s in answer_seq)

print(f'{num_words = }')
print(f'{q_max_len = }')
print(f'{a_max_len = }')

num_words = 43366
q_max_len = 57
a_max_len = 400


In [10]:
# 패딩: 질문이 훨씬 짧으므로 pre로 설정
question_input = pad_sequences(question_seq, maxlen=q_max_len, padding='pre')
answer_input = pad_sequences(answer_seq, maxlen=a_max_len, padding='post')

print(question_input.shape)
print(answer_input.shape)

print(question_input[1000])
print([tokenizer.index_word[s] for s in question_input[1000] if s != 0])
print(answer_input[1000])
print([tokenizer.index_word[s] for s in answer_input[1000] if s != 0])

(11926, 57)
(11926, 400)
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0  408    6
 5593   16  209  296   33  461    8   78   57    9 1955  461   35  493
  600   21   40   68    2    1 3668    6  109   50   20  138  182  150
   13]
['개발자', '가', '갖추어야', '할', '중요한', '역량', '과', '태도', '는', '무엇', '이라고', '생각', '하시고', '태도', '로', '인해서', '긍정', '적', '인', '경험', '을', '<UNK>', '케이스', '가', '있다면', '말씀', '해', '주', '시기', '바랍니다', '<eos>']
[ 408   53  209    7   15    1   57    9   26 2230   33 1744  464    3
 3898  117    2   32   16   12  107    9   61  887  980   78  152   34
  209    7   15    1   57    9   26  117  219    7   15 2554   28 1634
    5 2289   14    7   38  174   24    4 2230   33 1744  464   22 1388
  200    7    2  264    7   28  200    7    2  457 5791    7   15   56
  646   33 1658    5 5029 2726    7    3  117   38  174   24    4  117
    2   79  993    3   78  130 7827  993    2   64   17   48  993

In [11]:
class InterviewDataset(Dataset):
    def __init__(self, question_input, answer_input):
        self.question_input = torch.tensor(question_input, dtype=torch.long)
        self.answer_input = torch.tensor(answer_input, dtype=torch.long)
        
    def __len__(self):
        return len(self.question_input)
    
    def __getitem__(self, idx):
        return self.question_input[idx], self.answer_input[idx]

In [12]:
X_train, X_val, y_train, y_val = train_test_split(question_input, answer_input, test_size=0.2, random_state=42)

train_dataset = InterviewDataset(X_train, y_train)
val_dataset = InterviewDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, BATCH_SIZE)

print('학습 데이터 배치 개수:', len(train_loader))

학습 데이터 배치 개수: 150


In [13]:
# 인코더
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.gru = nn.GRU(embedding_dim, latent_dim, batch_first=True)

  def forward(self, input_seq):
    input_seq = self.embedding(input_seq)
    outputs, hidden = self.gru(input_seq)
    return hidden

In [14]:
# 디코더
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.gru = nn.GRU(embedding_dim, latent_dim, batch_first=True)
    self.fc = nn.Linear(latent_dim, vocab_size)

  def forward(self, input_seq, hidden):
    input_seq = self.embedding(input_seq)
    output, hidden = self.gru(input_seq, hidden)
    predict = self.fc(output)
    return predict, hidden

In [21]:
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# seq2seq
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder, device):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.device = device

  def forward(self, question, answer, teacher_forcing=0.5):
    batch_size = answer.shape[0]
    target_len = answer.shape[1]
    target_vocab_size = self.decoder.fc.out_features

    outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(self.device)
    hidden = self.encoder(question)

    input_first = answer[:, 0].unsqueeze(1)
    
    for target in range(1, target_len):
      output, hidden = self.decoder(input_first, hidden)
      outputs[:, target, :] = output.squeeze(1)

      teacher_force = random.random() < teacher_forcing
      top = output.argmax(2)

      input_first = answer[:, target].unsqueeze(1) if teacher_force else top

    return outputs

In [22]:
encoder = Encoder(num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(num_words, EMBEDDING_DIM, LATENT_DIM)

model = Seq2Seq(encoder, decoder, device).to(device)
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(43366, 300)
    (gru): GRU(300, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(43366, 300)
    (gru): GRU(300, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=43366, bias=True)
  )
)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.01)
epochs = 100

train_losses, train_accs, val_losses, val_accs = [], [], [], [] 

for epoch in range(epochs):
    model.train()
    train_loss, train_correct, train_tokens = 0, 0, 0

    for q_batch, a_batch in train_loader:
        q_batch, a_batch = q_batch.to(device), a_batch.to(device)
        
        optimizer.zero_grad()
        output = model(q_batch, a_batch)

        loss = criterion(output.view(-1, num_words), a_batch.view(-1))
        
        loss.backward()
        optimizer.step()

        preds = output.argmax(dim=-1)
        train_loss += loss.detach().cpu().item()
        mask = a_batch != 0
        correct = (preds == a_batch) & mask
        train_correct += correct.sum().detach().cpu().item()
        train_tokens += mask.sum().detach().cpu().item()

    train_loss /= len(train_loader)
    train_acc = train_correct / train_tokens
    train_losses.append(train_loss)
    train_accs.append(train_acc)

model.eval()
with torch.no_grad():
    val_loss, val_correct, val_tokens = 0, 0, 0

    for q_batch, a_batch in val_loader:
      q_batch = q_batch.to(device)
      a_batch = a_batch.to(device)

      output = model(q_batch, a_batch)
      output = output.view(-1, output.size(-1))
      a_batch = a_batch.view(-1)

      loss = criterion(output, a_batch)

      preds = output.argmax(dim=-1)
      val_loss += loss.detach().cpu().item()
      mask = a_batch != 0
      correct = (preds == a_batch) & mask
      val_correct += correct.sum().detach().cpu().item()
      val_tokens += mask.sum().detach().cpu().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens
    val_losses.append(val_loss)
    val_accs.append(val_acc)

print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')
     